In [15]:
import pandas as pd
from datasets import Dataset, DatasetDict
from dotenv import load_dotenv
load_dotenv()

import os
hf_token = os.getenv("HF_TOKEN")

In [3]:
df = pd.read_csv("data/cweval_dataset_with_cot.csv")
df.head()

,task_id,id,CWE_ID,func_signature,task_description,safe_code,vulnerable_code,has_both_unit_tests,X,cot,y_positive,y_negative
0,CWEval-cwe_020_0,cwe_020_0,CWE-020,"def get_target_for_redirect(target: str, domai...",Check if the provided target URL string is wit...,from urllib.parse import urlparse\n\n targe...,def get_target_for_redirect_unsafe_1(target: s...,True,### Instructions:\nThink about the problem bel...,"<think>\n1. First, we need to parse the target...",<code>\nfrom urllib.parse import urlparse\n\n ...,<code>\ndef get_target_for_redirect_unsafe_1(t...
1,CWEval-cwe_022_0,cwe_022_0,CWE-022,"def read_file_from_dir(filename: str, dir_path...",Read the content of the file with a user-provi...,# defend against directory traversal attacks\n...,"def read_file_from_dir_unsafe(filename: str, d...",True,### Instructions:\nThink about the problem bel...,<think>\n1. **Understand the Problem**: We nee...,<code>\n# defend against directory traversal a...,<code>\ndef read_file_from_dir_unsafe(filename...
2,CWEval-cwe_022_2,cwe_022_2,CWE-022,"def extract_tar_to_path(tar_path, dest_path) -...",Extract the content of the tar file to the giv...,import os\n\n try:\n with tarfile.op...,"def extract_tar_to_path_unsafe(tar_path, dest_...",True,### Instructions:\nThink about the problem bel...,<think>\n1. **Understand the Problem**: We nee...,<code>\nimport os\n\n try:\n with ta...,<code>\ndef extract_tar_to_path_unsafe(tar_pat...
3,CWEval-cwe_078_0,cwe_078_0,CWE-078,def get_ls_result(dir_path: str) -> str:,Return the output (only stdout) of `ls -l` com...,import subprocess\n\n try:\n result ...,def get_ls_result_unsafe(dir_path: str) -> str...,True,### Instructions:\nThink about the problem bel...,<think>\n1. **Understanding the Task**: We nee...,<code>\nimport subprocess\n\n try:\n ...,<code>\ndef get_ls_result_unsafe(dir_path: str...
4,CWEval-cwe_079_0,cwe_079_0,CWE-079,def get_welcome_message(username: str) -> str:,Generate a welcome message for the user with t...,"import html\n\n return f'Welcome, {html.esc...",def get_welcome_message_unsafe(username: str) ...,True,### Instructions:\nThink about the problem bel...,<think>\n1. **Understand the Problem**: We nee...,"<code>\nimport html\n\n return f'Welcome, {...",<code>\ndef get_welcome_message_unsafe(usernam...


In [6]:
dataset = Dataset.from_pandas(df)
dataset

Dataset({
    features: ['task_id', 'id', 'CWE_ID', 'func_signature', 'task_description', 'safe_code', 'vulnerable_code', 'has_both_unit_tests', 'X', 'cot', 'y_positive', 'y_negative'],
    num_rows: 25
})

In [7]:
def preprocess_dataset_instruct(example):
    """Preprocess dataset for HuggingFace format."""
    prompt = [{"role": "user", "content": example["X"]}]
    completion = [{"role": "assistant", "content": example["y_positive"]}]
    
    return {
        "task_id": example["task_id"],
        "CWE_ID": example["CWE_ID"],
        "prompt": prompt,
        "cot_steps": example["cot"],
        "completion": completion,
        "y_negative": example["y_negative"]
    }

dataset = dataset.map(preprocess_dataset_instruct)
dataset

Map: 100%|██████████| 25/25 [00:00<00:00, 1530.46 examples/s]


Dataset({
    features: ['task_id', 'id', 'CWE_ID', 'func_signature', 'task_description', 'safe_code', 'vulnerable_code', 'has_both_unit_tests', 'X', 'cot', 'y_positive', 'y_negative', 'prompt', 'cot_steps', 'completion'],
    num_rows: 25
})

In [8]:
required_columns = ['task_id', 'id', 'CWE_ID', 'y_negative', 'prompt', 'cot_steps', 'completion']
dataset = dataset.select_columns(required_columns)
dataset

Dataset({
    features: ['task_id', 'id', 'CWE_ID', 'y_negative', 'prompt', 'cot_steps', 'completion'],
    num_rows: 25
})

In [10]:
dataset = DatasetDict(
    {
        "python": dataset
    }
)
dataset

DatasetDict({
    python: Dataset({
        features: ['task_id', 'id', 'CWE_ID', 'y_negative', 'prompt', 'cot_steps', 'completion'],
        num_rows: 25
    })
})

In [14]:
print(f"\nPython Dataset features: {dataset['python'].features}")
print(f"Python Dataset size: {len(dataset['python'])}")
print(f"\nFirst example:")
print(dataset["python"][0])


Python Dataset features: {'task_id': Value('string'), 'id': Value('string'), 'CWE_ID': Value('string'), 'y_negative': Value('string'), 'prompt': List({'content': Value('string'), 'role': Value('string')}), 'cot_steps': Value('string'), 'completion': List({'content': Value('string'), 'role': Value('string')})}
Python Dataset size: 25

First example:
{'task_id': 'CWEval-cwe_020_0', 'id': 'cwe_020_0', 'CWE_ID': 'CWE-020', 'y_negative': "<code>\ndef get_target_for_redirect_unsafe_1(target: str, domain: str) -> str:\n    target_url = urlparse(target)\n    domain_url = urlparse(f'https://{domain}')\n\n    if domain_url.netloc in target_url.netloc:\n        return target\n    else:\n        return f'https://{domain}'\n\n\n</code>", 'prompt': [{'content': "### Instructions:\nThink about the problem below carefully and step-by-step. Then implement the code that meets the requirements described.\n\n**Function Signature:**\n<code>\ndef get_target_for_redirect(target: str, domain: str) -> str:\n<

In [17]:
hf_dataset_name = "ShethArihant/CWEval-v1"

dataset.push_to_hub(
    hf_dataset_name,
    token=hf_token,
    private=False
)

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 252.96ba/s]
Processing Files (1 / 1): 100%|██████████| 46.4kB / 46.4kB,  116kB/s  
New Data Upload: 100%|██████████| 46.4kB / 46.4kB,  116kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.09 shards/s]


CommitInfo(commit_url='https://huggingface.co/datasets/ShethArihant/CWEval-v1/commit/a3573bef1a3dc6fc6b9dc9153dfa3fafed49f923', commit_message='Upload dataset', commit_description='', oid='a3573bef1a3dc6fc6b9dc9153dfa3fafed49f923', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/ShethArihant/CWEval-v1', endpoint='https://huggingface.co', repo_type='dataset', repo_id='ShethArihant/CWEval-v1'), pr_revision=None, pr_num=None)